# 00 — Setup & Data

Mount Drive, install dependencies, fetch benchmark targets, sanity-check GPU.
**Run this first.** Expected time on T4: ~15 min (downloads dominate).

In [ ]:
# Drive
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/rfd3-vs-chroma'
os.makedirs(ROOT, exist_ok=True)
os.environ['RFD3_CHROMA_ROOT'] = ROOT
print('ROOT =', ROOT)

In [ ]:
# Clone the repo (replace YOUR_USERNAME with your fork)
REPO = 'YOUR_USERNAME/rfd3-vs-chroma'
import os, subprocess
if not os.path.exists('/content/repo'):
    subprocess.run(['git', 'clone', '-q', f'https://github.com/{REPO}.git', '/content/repo'])
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

In [ ]:
# Light deps (used by every notebook)
!pip install -q biotite biopython rdkit py3Dmol scipy pandas matplotlib tabulate

In [ ]:
from utils import gpu_info
print(gpu_info())

## RFdiffusion3 install (foundry)

In [ ]:
# rc-foundry installs the `rfd3` CLI (and `mpnn`/`rf3` if you want them)
!pip install -q "rc-foundry[rfd3]"

In [ ]:
# Download RFD3 weights to ~/.foundry/checkpoints (or set FOUNDRY_CHECKPOINT_DIRS)
!foundry install rfd3 || echo "If this fails, see docs/TROUBLESHOOTING.md"

In [ ]:
# Smoke test: rfd3 CLI on PATH
!which rfd3 && rfd3 --help 2>&1 | head -10 || echo "rfd3 not on PATH yet"

## LigandMPNN install (standalone — stable API)

Foundry's `mpnn` reimplementation is still under active development. We use the
stable upstream LigandMPNN repo for sequence design.

In [ ]:
from utils import ensure_ligandmpnn
print('LigandMPNN ready:', ensure_ligandmpnn())

## Chroma install

In [ ]:
!pip install -q generate-chroma

In [ ]:
# Free key at https://chroma-weights.generatebiomedicines.com/
import os, getpass
if not os.environ.get('CHROMA_API_KEY'):
    os.environ['CHROMA_API_KEY'] = getpass.getpass('Chroma API key: ')

from chroma import api
api.register_key(os.environ['CHROMA_API_KEY'])

In [ ]:
# Verify Chroma loads (downloads weights on first call, ~1 GB)
from chroma import Chroma
chroma = Chroma()
print('Chroma backbone:', type(chroma.backbone_network).__name__)

## Fetch benchmark targets

In [ ]:
from utils import fetch_pdb, DATA

DNA_TARGETS = ['7m5w', '7rte', '7n5u']                 # RFD3 §3.3
LIGAND_TARGETS = {'FAD': '7bkc', 'OQO': '7v11',
                  'IAI': '5sdv', 'SAM': '7c7m'}        # RFD3 §3.4
ENZYME_REFS = ['1euv']                                  # Ulp1
PPI_TARGETS = {'PD-L1': '5o45', 'InsulinR': '4zxb',
               'IL-7Ra': '3di3', 'Tie2': '2gy5',
               'IL-2Ra': '1z92'}                       # RFD3 §3.2

all_pids = DNA_TARGETS + list(LIGAND_TARGETS.values()) + ENZYME_REFS + list(PPI_TARGETS.values())
for pid in all_pids:
    try:
        p = fetch_pdb(pid)
        print(f'{pid}: {p.stat().st_size // 1024} KB')
    except Exception as e:
        print(f'{pid}: FAILED ({e})')

In [ ]:
# Optional: 10 AlphaFoldDB structures for novelty comparison
import urllib.request, socket
socket.setdefaulttimeout(30)

afdb_dir = DATA / 'afdb_samples'
afdb_dir.mkdir(exist_ok=True)
for uid in ['P0DTC2', 'P00533', 'P04637', 'P10275', 'P05067',
            'P02768', 'P01308', 'P00734', 'P15056', 'P38398']:
    out = afdb_dir / f'{uid}.pdb'
    if out.exists():
        continue
    try:
        urllib.request.urlretrieve(
            f'https://alphafold.ebi.ac.uk/files/AF-{uid}-F1-model_v4.pdb', out)
    except Exception:
        pass
print('AFDB:', len(list(afdb_dir.glob('*.pdb'))), 'structures')

In [ ]:
import json
from utils import RESULTS

manifest = {
    'dna_targets': DNA_TARGETS,
    'ligand_targets': LIGAND_TARGETS,
    'enzyme_refs': ENZYME_REFS,
    'ppi_targets': PPI_TARGETS,
    'gpu': gpu_info(),
}
(RESULTS / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

**Done.** Proceed to `01_unconditional_benchmark.ipynb`.

If `foundry install rfd3` failed, see `docs/TROUBLESHOOTING.md`.